In [4]:
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
import numpy as np
from typing import List
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [6]:
import tempfile

temp_dir = tempfile.mkdtemp()
for doc_num, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{doc_num}.txt", "w") as f:
        f.write(doc)

print(f"Sample files created in directory {temp_dir}")

Sample files created in directory C:\Users\sumsaras\AppData\Local\Temp\tmpvjzfgzu_


### Creating Sample Files

In [7]:
import tempfile

temp_dir = tempfile.mkdtemp()
for doc_num, doc in enumerate(sample_docs):
    with open(f"doc_{doc_num}.txt", "w") as f:
        f.write(doc)

print(f"Sample files created in directory {temp_dir}")

Sample files created in directory C:\Users\sumsaras\AppData\Local\Temp\tmpqfx92z6i


### Loading Documents

In [8]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()
print(documents)
print(f"Total Documents : {len(documents)}")


[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '), Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natura

### Splitting documents

In [9]:
recursiveTextSplitter = RecursiveCharacterTextSplitter(
    chunk_size= 100,
    chunk_overlap= 50,
    separators=[" ","\n"]
)

chunks = recursiveTextSplitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print(chunks)

for num, chunk in enumerate(chunks):
    print(f"Chunk_{num} : \n {chunk.page_content}")

Total chunks: 30
[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that'), Document(metadata={'source': 'data\\doc_0.txt'}, page_content='is a subset of artificial intelligence that enables systems to learn \n    and improve from'), Document(metadata={'source': 'data\\doc_0.txt'}, page_content='enables systems to learn \n    and improve from experience without being explicitly programmed.'), Document(metadata={'source': 'data\\doc_0.txt'}, page_content='experience without being explicitly programmed. There are three main \n    types of machine'), Document(metadata={'source': 'data\\doc_0.txt'}, page_content='There are three main \n    types of machine learning: supervised learning, unsupervised learning,'), Document(metadata={'source': 'data\\doc_0.txt'}, page_content='supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning'), Document(met

### Embeddings

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings()
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### Ingesting embeddings into vector store

In [11]:

persistent_db_dir = "./chroma_db_dir"
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model = "sentence-transformers/all-MiniLM-L6-v2"),
    persist_directory=persistent_db_dir,
    collection_name="rag_collection"
)

print(f"Vector store created with collection count {vector_store._collection.count()}")
print(f"Persisted to {persistent_db_dir}")

Vector store created with collection count 87
Persisted to ./chroma_db_dir


In [12]:
query = "What is machine learning?"
results = vector_store.similarity_search(query, k=3)
results

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n   

In [13]:
query = "What is deep learning?"
results = vector_store.similarity_search(query, k=3)
results

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, 

In [14]:
query = "What is NLP"
results = vector_store.similarity_search_with_score(query, k = 10)
results

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.637130856513977),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These 

In [15]:
query = "What is NLP"
results = vector_store.similarity_search_with_relevance_scores(query, k = 3)
results

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.5494804508557736),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These

In [16]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(
    model="hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF"
)
messages = [
    (
        "system",
        "You are expert in general purpose knowledge, answer my questions in a polite way",
    ),
    ("human", "What is OPENAI"),
]
ai_msg = llm.invoke(messages)
ai_msg

'You\'re interested in learning about artificial intelligence and its applications. OpenAI is indeed an extremely respected organization that focuses on developing and deploying cutting-edge AI technologies.\n\nOpenAI was founded in 2015 by a group of researchers and engineers, led by Elon Musk, who are also the CEO of SpaceX. The name "OpenAI" comes from the Russian phrase "open-source," which emphasizes the organization\'s commitment to making its software and technology accessible to anyone, without restriction.\n\nOne of OpenAI\'s most notable achievements is the development of its model, called AI (as in "artificial intelligence"). This refers to a type of machine learning algorithm that enables computers to learn from data and improve their performance over time. The AI system has achieved significant success in various applications, including natural language processing, computer vision, and reinforcement learning.\n\nSome of OpenAI\'s notable projects include:\n\n1. **Language 

### Moder RAG Chain

In [2]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

c:\Users\sumsaras\Documents\PersonalWorkspace\AI_Learning\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
vector_retreiver = vector_store.as_retriever(search_args={"k":3})
vector_retreiver

NameError: name 'vector_store' is not defined

In [1]:
system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

context: {context}
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
    # ("ai", "Hi there, how can I help you?")
])

prompt_template


NameError: name 'ChatPromptTemplate' is not defined

In [33]:
### Now in this step both llm model and the prompt_template that created above is chained together, the responses fetched
### from the vector store will provide to the prompt inside the context and then ai will respond based on that
document_chain = create_stuff_documents_chain(llm, prompt_template)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| OllamaLLM(model='hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, c

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

In [ ]:
rag_chain = create_retrieval_chain(vector_retreiver, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E51DBD2490>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you

In [ ]:
ai_msg = rag_chain.invoke({"input": "What is deep learning"})
ai_msg

{'input': 'What is deep learning',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like comp

In [ ]:
ai_msg['answer']

"Deep learning is a subset of machine learning based on artificial neural networks that were inspired by the human brain's structure. These networks consist of layers of interconnected nodes to process data efficiently in fields such as computer vision, natural language processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly effective for image processing."

In [ ]:
ai_msg = rag_chain.invoke({"input":"What is machine learning"})
ai_msg['answer']

'Machine Learning is a subset of artificial intelligence that enables systems to learn from experience without being explicitly programmed.\n\nIt involves three main types of machine learning:\n\n1. Supervised learning uses labeled data to train models.\n2. Unsupervised learning finds patterns in unlabeled data.\n3. Reinforcement learning learns through interaction with an environment using rewards and penalties.'

In [ ]:
ai_msg = rag_chain.invoke({"input": "what is the purpose of machine learning"})
ai_msg['answer']

'Machine learning aims to enable systems to learn from experience without being explicitly programmed, enabling them to improve over time and make predictions or decisions based on new data.'

In [ ]:
ai_msg = rag_chain.invoke({"input": "How deeplearning, machine learning and nueral language programming connected to each other"})
ai_msg

{'input': 'How deeplearning, machine learning and nueral language programming connected to each other',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'),
  Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep 

### Langchain Expression Language(LCEL)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [ ]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}
""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n"), additional_kwargs={})])

In [ ]:
def format_docs(docs):
    return "####".join(doc.page_content for doc in docs)

In [ ]:
expression = (
    {
        "context" : vector_retreiver | format_docs,
        "question" : RunnablePassthrough()
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

expression

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E51DBD2490>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n"), additional_kwargs={})])
| OllamaLLM(model='hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF')
| StrOutputParser()

In [ ]:
response = expression.invoke("What is deep learning")
response

'According to the context, "Deep Learning" refers to a subset of machine learning based on artificial neural networks. \n\nSpecific details from the context that support this answer:\n\n1. It mentions "Convolutional Neural Networks (CNNs)" which are an example of deep learning.\n2. It states "Recurrent Neural Networks (RNNs) and Transformers" which also fall under the category of deep learning.\n\nTherefore, based on these examples and explanations in the context, it can be inferred that "Deep Learning" is indeed a subset of machine learning based on artificial neural networks, specifically those consisting of CNNs.'

### Adding new documents to vector store

In [ ]:
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [ ]:
from langchain_core.documents import Document
document = Document(
    page_content= new_document,
    metadata = {"topic":"Reinforment Learning", "source":"New document"}   
)
document

Document(metadata={'topic': 'Reinforment Learning', 'source': 'New document'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [ ]:
new_chunks = recursiveTextSplitter.split_documents([document])
new_chunks

[Document(metadata={'topic': 'Reinforment Learning', 'source': 'New document'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where'),
 Document(metadata={'topic': 'Reinforment Learning', 'source': 'New document'}, page_content='learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting'),
 Document(metadata={'topic': 'Reinforment Learning', 'source': 'New document'}, page_content='an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards'),
 Document(metadata={'topic': 'Reinforment Learning', 'source': 'New document'}, page_content='with an environment. The agent receives rewards or penalties \nbased on its actions and learns to'),
 Document(metadata={'topic': 'Reinforment Learning', 'source': 'New document'}, page_content='or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts'),
 Document(metada

In [ ]:
vector_store.add_documents(new_chunks)

['2a791840-6d13-4c6e-a2cc-45e6bbd6897b',
 '1767570e-6bbe-45d4-b332-f3c3207151da',
 'd932983b-b898-42a1-804a-5f0ca012141e',
 'a57fb01f-1634-4e81-86ff-586eb9b6a294',
 '01b9882f-aba8-4c1c-8d07-a96bf6478867',
 'e355c369-a5c0-411b-8d1d-feee5a0d0d54',
 '92d1d8df-2c94-4d43-b0f0-f44e3c94df6e',
 '82ae6b9d-a969-471b-b2ee-78d4c9c221c6',
 '1e15f129-8199-4b5e-bda2-d7dd8ba54a44',
 'f828361e-3e36-4058-828b-b069ae919d34',
 '55aaf1c9-518a-4606-ae42-2094000fd115']

In [ ]:
count = vector_store._collection.count()
count

57

### History Aware RAG - Conversational Memory

Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

In [34]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

In [74]:
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

history_system_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])


In [75]:
history_aware_retriever = create_history_aware_retriever(
    llm,
    vector_retreiver,
    history_system_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001B6BCD8CC20>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag

In [78]:
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {input}
""")

qa_document_chain = create_stuff_documents_chain(llm, custom_prompt)
qa_retrievel_chain = create_retrieval_chain(history_aware_retriever, qa_document_chain)

qa_retrievel_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001B6BCD8CC20>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')],

In [79]:
chat_history = []
result = qa_retrievel_chain.invoke({
    "chat_history" : chat_history,
    "input": "What is Deep Learning"
})
print("Q. What is Deep Learning?")
print(result)
result['answer']

Q. What is Deep Learning?
{'chat_history': [], 'input': 'What is Deep Learning', 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers \n    excel at sequential data processing.'), Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep l

'Based on the context provided, I would say that:\n\nDeep Learning refers to a subset of machine learning based on artificial neural networks.\n\nThe text states that "These networks are inspired by the human brain... Consisting of layers of interconnected nodes." This suggests that deep learning shares similarities with artificial neural networks, which are also inspired by the human brain.'

In [81]:
chat_history.extend([
    HumanMessage(content="What is Deep Learning"),
    AIMessage(content=result['answer'])
])
followup_result = qa_retrievel_chain.invoke({
    "chat_history": chat_history,
    "input" : "What are its types"
})

print(followup_result)
followup_result['answer']

{'chat_history': [HumanMessage(content='What is Deep Learning', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the context provided, I would say that:\n\nDeep Learning refers to a subset of machine learning based on artificial neural networks.\n\nThe text states that "These networks are inspired by the human brain... Consisting of layers of interconnected nodes." This suggests that deep learning shares similarities with artificial neural networks, which are also inspired by the human brain.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is Deep Learning', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the context provided, I would say that:\n\nDeep Learning refers to a subset of machine learning based on artificial neural networks.\n\nThe text states that "These networks are inspired by the human brain... Consisting of layers of interconnected nodes." This suggests that 

'Since I don\'t have any specific information or context about "artificial neural networks", it seems like the question is referring to a general topic related to AI and machine learning.\n\nHowever, according to my research, there are several types of artificial neural networks:\n\n1. **Feedforward Neural Networks**: These are the simplest type of neural network, where data flows only in one direction through multiple layers.\n2. **Recurrent Neural Networks (RNNs)**: These networks process data sequentially, allowing them to keep track of previous information.\n3. **Convolutional Neural Networks (CNNs)**: These networks use convolutional and pooling operations to analyze data in a spatial manner, making them particularly well-suited for image and video processing.\n4. **Generative Adversarial Networks (GANs)**: These networks are used to generate new data that resembles existing data, often in the form of images or text.\n5. **Long Short-Term Memory (LSTM) Neural Networks**: A type of

In [80]:
chat_history.extend([
    HumanMessage(content="What is Deep Learning"),
    AIMessage(content=result['answer'])
])
followup_result = qa_retrievel_chain.invoke({
    "chat_history": chat_history,
    "input" : "Why sun is so far"
})

print(followup_result)
followup_result['answer']

{'chat_history': [HumanMessage(content='What is Deep Learning', additional_kwargs={}, response_metadata={}), AIMessage(content='Based on the context provided, I would say that:\n\nDeep Learning refers to a subset of machine learning based on artificial neural networks.\n\nThe text states that "These networks are inspired by the human brain... Consisting of layers of interconnected nodes." This suggests that deep learning shares similarities with artificial neural networks, which are also inspired by the human brain.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'input': 'Why sun is so far', 'context': [Document(metadata={'source': 'data\\doc_2.txt'}, page_content='is a field of AI that focuses on the interaction between computers and human language. \n    Key'), Document(metadata={'source': 'data\\doc_2.txt'}, page_content='is a field of AI that focuses on the interaction between computers and human language. \n    Key'), Document(metadata={'sour

'I don\'t know.\n\nThe context doesn\'t provide any information about why the sun is "so far". The only relevant details are:\n\n* Sun: This refers to the sun, which is a celestial body and not an object with size or distance.\n* So far: This indicates that there may be a reference point or context where the question asks about the sun\'s distance.\n\nFrom this information, it\'s unclear why the sun would be "so far".'